# 02 — VAR dynamics and rolling residuals

Fit the interpretable conditional-mean model and construct pseudo-out-of-sample calibration errors. The test period remains untouched.

In [5]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from innovcal.data import chronological_split
from innovcal.di_var import rolling_var_residuals
from innovcal.innovations.diagnostics import summarize_innovations
from innovcal.vector_ar.fit import fit_var_ols
from innovcal.vector_ar.stability import stability_summary

In [6]:
returns = pd.read_csv(ROOT / 'data/processed/financial_returns.csv', index_col=0, parse_dates=True)
split = chronological_split(returns.to_numpy(), 0.6, 0.2)
LAGS = 1
calibration_sample = np.vstack([split.train, split.calibration])
residuals = rolling_var_residuals(calibration_sample, len(split.train), lags=LAGS)
var_fit = fit_var_ols(calibration_sample, lags=LAGS, include_intercept=True)
print('train/calibration/test:', len(split.train), len(split.calibration), len(split.test))
print('rolling residuals:', residuals.shape)

/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: divide by zero encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: overflow encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: invalid value encountered in matmul
  fitted = X @ beta


train/calibration/test: 2867 955 957
rolling residuals: (955, 4)


In [7]:
k = returns.shape[1]
beta_no_intercept = var_fit['beta'][1:]
print(stability_summary(beta_no_intercept, k=k, lags=LAGS))
diagnostics = summarize_innovations(residuals)
correlation = diagnostics.pop('correlation')
display(pd.DataFrame(diagnostics, index=returns.columns))
display(pd.DataFrame(correlation, index=returns.columns, columns=returns.columns))

{'stable': True, 'eigenvalues': array([-0.15831036+0.j        ,  0.04287749+0.j        ,
       -0.01592224+0.02712515j, -0.01592224-0.02712515j]), 'max_modulus': 0.15831035897856838}


,mean,std,skewness,kurtosis,excess_kurtosis,jarque_bera_stat,jarque_bera_pvalue
AAPL,0.000336,0.020892,-0.442602,8.636020,5.636020,1279.073401,1.789534e-278
JPM,-0.000090,0.020871,-0.205266,14.601170,11.601170,5300.519804,0.000000e+00
XOM,0.000156,0.021661,-0.150240,9.094868,6.094868,1463.297370,0.000000e+00
WMT,0.000284,0.013569,0.659872,14.522792,11.522792,5291.595399,0.000000e+00


,AAPL,JPM,XOM,WMT
AAPL,1.000000,0.442588,0.341120,0.383144
JPM,0.442588,1.000000,0.655864,0.266215
XOM,0.341120,0.655864,1.000000,0.217579
WMT,0.383144,0.266215,0.217579,1.000000


In [8]:
CACHE = ROOT / 'results/notebook_cache'
CACHE.mkdir(parents=True, exist_ok=True)
np.savez(
    CACHE / 'var_stage.npz',
    train=split.train, calibration=split.calibration, test=split.test,
    residuals=residuals, beta=var_fit['beta'], lags=LAGS,
)